# Module 4 — Exercises (Student Worksheet)
### Equivariant Graph Networks: EGNN, LorentzNet & L-GATr · TIFR ML School 2026

Student worksheet for the seven exercises at the end of
`Module4_Equivariant_EGNN_LorentzNet.ipynb`. **Self-contained**: the *Setup* re-defines the Minkowski
scalarization block (LGEB), LorentzNet-lite, the DeepSets baseline, the Lorentz-transform helpers, and loads
JetClass as sets of 4-momenta.

| # | Exercise | Idea it drives home |
|---|----------|---------------------|
| 1 | **Break the symmetry** | feeding one non-invariant (E) destroys the exact guarantee |
| 2 | **Augment harder** | augmentation only *approximates* invariance, and pays in epochs |
| 3 | **Frame-dependent graph** | a k-NN on an *invariant* metric is safe; on a lab metric it breaks |
| 4 | **PELICAN-style** | use *all* pairwise invariants + a permutation-equivariant aggregator |
| 5 | **Equivariant regression** | an invariant net *cannot* output a 4-vector; LGEB's coordinate branch can |
| 6 | **Will a big model catch up?** | capacity vs a correct inductive bias, as data grows |
| 7 | **Grow the L-GATr** | add the geometric bilinear + scalar channel; does AUC rise, invariance hold? |

> **Speed knobs:** `N_PER_CLS`, `EPOCHS`, and the per-exercise train sizes. LorentzNet is the runtime
> bottleneck (fully-connected graph over 4-vectors); defaults are sized to finish in a few minutes on a laptop.

> **How to use this worksheet.** Run the **Setup** section as-is (it loads the data and provides the models/helpers from the module). Then complete each exercise where you see `# TODO` and `raise NotImplementedError`. Read the **prompt** above each exercise — it sketches the approach. Check your work against the matching `*_Exercises_Solutions.ipynb`.

## Setup — scalarization block, LorentzNet, Lorentz transforms, data

In [ ]:
import os, math, time, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torch_geometric.utils import scatter
from torch_geometric.nn import global_mean_pool, global_add_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score

torch.manual_seed(0); np.random.seed(0)
device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("device", device)

def mlp(i, o, h=64):
    return nn.Sequential(nn.Linear(i, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, o))

def fc_edge_index(batch):
    """Fully-connected edges WITHIN each graph (frame-independent). edge_index [2,E]=[src j, tgt i]."""
    same = batch.unsqueeze(0) == batch.unsqueeze(1); same.fill_diagonal_(False)
    idx = same.nonzero(as_tuple=False)
    return torch.stack([idx[:, 1], idx[:, 0]], dim=0)

def random_rotation(dim=3):
    A = torch.randn(dim, dim); Q, _ = torch.linalg.qr(A)
    if torch.det(Q) < 0: Q[:, 0] = -Q[:, 0]
    return Q

In [ ]:
# --- Minkowski scalarization: LGEB + LorentzNet-lite (from Module 4) ---------------------
def minkowski(a, b):                       # metric (+,-,-,-)
    return a[..., 0]*b[..., 0] - (a[..., 1:]*b[..., 1:]).sum(-1)
def psi(s):                                # sign-preserving log squash of huge dynamic range
    return torch.sign(s) * torch.log(torch.abs(s) + 1.0)

class LGEB(nn.Module):
    def __init__(self, hidden, c=0.01, update_coords=True):
        super().__init__()
        self.phi_e = mlp(2*hidden + 2, hidden); self.phi_h = mlp(2*hidden, hidden)
        self.phi_x = nn.Linear(hidden, 1); self.norm = nn.LayerNorm(hidden)
        self.c, self.update_coords = c, update_coords
    def forward(self, h, x, edge_index):
        src, dst = edge_index; diff = x[dst] - x[src]
        norm2 = psi(minkowski(diff, diff)).unsqueeze(-1)
        ip    = psi(minkowski(x[dst], x[src])).unsqueeze(-1)
        m = self.phi_e(torch.cat([h[dst], h[src], norm2, ip], dim=-1))
        agg = scatter(m, dst, dim=0, dim_size=h.size(0), reduce="mean")
        h = self.norm(h + self.phi_h(torch.cat([h, agg], dim=-1)))
        if self.update_coords:
            x = x + self.c * scatter(self.phi_x(m)*diff, dst, dim=0, dim_size=x.size(0), reduce="mean")
        return h, x

class LorentzNetLite(nn.Module):
    def __init__(self, hidden=64, n_layers=3, n_classes=2):
        super().__init__()
        self.embed = nn.Linear(2, hidden)
        self.blocks = nn.ModuleList([LGEB(hidden, update_coords=False) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, n_classes))
    def forward(self, data):
        x, batch = data.pos, data.batch
        p_jet = global_add_pool(x, batch)[batch]
        h = self.embed(torch.stack([psi(minkowski(x, x)), psi(minkowski(x, p_jet))], dim=-1))
        edge_index = fc_edge_index(batch)
        for blk in self.blocks: h, x = blk(h, x, edge_index)
        return self.head(global_mean_pool(h, batch))

class DeepSetsBaseline(nn.Module):
    """Permutation-invariant but NOT Lorentz-aware: a DeepSet on raw 4-vector components."""
    def __init__(self, hidden=64, n_classes=2):
        super().__init__(); self.phi = mlp(4, hidden)
        self.head = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, n_classes))
    def forward(self, data):
        return self.head(global_mean_pool(self.phi(data.x), data.batch))

In [ ]:
# --- Lorentz transforms (to hit jets with) ----------------------------------------------
def rotation4(R3): L = torch.eye(4); L[1:, 1:] = R3; return L
def boost_matrix(beta):
    beta = torch.as_tensor(beta, dtype=torch.float32); b2 = (beta*beta).sum()
    gamma = 1.0/torch.sqrt(1-b2); L = torch.eye(4)
    L[0, 0] = gamma; L[0, 1:] = gamma*beta; L[1:, 0] = gamma*beta
    L[1:, 1:] = torch.eye(3) + (gamma-1)*torch.outer(beta, beta)/b2
    return L
def boost_z(xi):
    c, s = math.cosh(xi), math.sinh(xi); L = torch.eye(4); L[0,0]=c; L[0,3]=s; L[3,0]=s; L[3,3]=c; return L
def random_lorentz(max_beta=0.8):
    b = torch.randn(3); b = b/b.norm()*(torch.rand(1)*max_beta)
    return rotation4(random_rotation()) @ boost_matrix(b) @ rotation4(random_rotation())
GMETRIC = torch.diag(torch.tensor([1., -1., -1., -1.]))
_L = random_lorentz()
print("random Λ preserves metric:", (_L.T @ GMETRIC @ _L - GMETRIC).abs().max().item() < 1e-4)

In [ ]:
# --- data: each jet = set of leading-N 4-momenta ----------------------------------------
import uproot, awkward as ak
CANDIDATE_PATHS = [
    "../data/JetClass_example_100k.root",
    "/Users/sanmay/Documents/ML_SCHOOLS/MLSCHOOL_2023_ICTS/Main_School/JetDataset/JetClass_example_100k.root",
    "./JetClass_example_100k.root",
]
root_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if root_path is None: raise FileNotFoundError("JetClass_example_100k.root not found")
print("Using:", root_path)

NPART, N_PER_CLS, SCALE, EPOCHS = 24, 4000, 0.01, 22
tree = uproot.open(root_path)["tree"]
br = tree.arrays(["part_px","part_py","part_pz","part_energy","label_QCD","label_Tbqq","label_Tbl"])
is_qcd = ak.to_numpy(br["label_QCD"]).astype(bool)
is_top = ak.to_numpy(br["label_Tbqq"]).astype(bool) | ak.to_numpy(br["label_Tbl"]).astype(bool)
sel = np.concatenate([np.where(is_qcd)[0][:N_PER_CLS], np.where(is_top)[0][:N_PER_CLS]])
labels = np.concatenate([np.zeros(N_PER_CLS), np.ones(N_PER_CLS)]).astype(np.int64)
px,py,pz,e = br["part_px"][sel], br["part_py"][sel], br["part_pz"][sel], br["part_energy"][sel]
order = ak.argsort(e, axis=1, ascending=False)
def topN(a): return ak.to_numpy(ak.fill_none(ak.pad_none(a[order], NPART, clip=True), 0.0))
P4 = (np.stack([topN(e), topN(px), topN(py), topN(pz)], axis=-1).astype(np.float32)) * SCALE   # (Njet,N,4)=(E,px,py,pz)
counts = np.minimum(ak.to_numpy(ak.num(e, axis=1)), NPART)

data_list = []
for i in range(len(labels)):
    nn_ = int(counts[i])
    if nn_ < 2: continue
    fm = torch.from_numpy(P4[i, :nn_])
    data_list.append(Data(x=fm, pos=fm, y=torch.tensor([int(labels[i])], dtype=torch.long)))
import random; random.seed(0); random.shuffle(data_list)
test_set = data_list[-3000:]; pool = data_list[:-3000]
test_loader = DataLoader(test_set, batch_size=128)
print(f"{len(data_list)} jets | pool {len(pool)} | test {len(test_set)} | <= {NPART} 4-vectors/jet")

In [ ]:
# --- train / eval helpers, and reference LorentzNet + DeepSets reused by several exercises ---
@torch.no_grad()
def auc_of(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(device); ys.append(d.y.cpu()); ps.append(F.softmax(model(d), -1)[:, 1].cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    if not np.isfinite(p).all():                              # graceful: don't crash the whole notebook
        print("  [warn] non-finite predictions -> clamped to 0.5 (model likely diverged; lower lr / more clip)")
        p = np.nan_to_num(p, nan=0.5, posinf=1.0, neginf=0.0)
    return roc_auc_score(y, p)

def train(model, loader, epochs=EPOCHS, lr=1e-3, augment=False):
    model = model.to(device); opt = torch.optim.AdamW(model.parameters(), lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for _ in range(epochs):
        model.train()
        for d in loader:
            d = d.to(device)
            if augment:
                L = random_lorentz().to(device); d = d.clone(); d.x = d.x @ L.T; d.pos = d.pos @ L.T
            loss = F.cross_entropy(model(d), d.y); opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)      # insurance against divergence
            opt.step()
        sched.step()
    return model

def n_params(m): return sum(p.numel() for p in m.parameters())

REF_N = 2000
ref_loader = DataLoader(pool[:REF_N], batch_size=128, shuffle=True)
torch.manual_seed(0); ref_lnet = train(LorentzNetLite(hidden=64, n_layers=3).to(device), ref_loader)
torch.manual_seed(0); ref_dset = train(DeepSetsBaseline(hidden=64).to(device), ref_loader)
print(f"reference @ {REF_N} jets:  LorentzNet AUC {auc_of(ref_lnet, test_loader):.3f}   "
      f"DeepSets AUC {auc_of(ref_dset, test_loader):.3f}")
print("Setup complete.")

## Exercise 1 — Break the symmetry

> *Feed the raw energy E_i (a non-invariant) into LorentzNet's node init alongside the mass. Re-run the §5b
> boost test — how badly is invariance violated, and what happens to small-data accuracy?*

**The concept.** LorentzNet is invariant because **every** input to its MLPs is a Lorentz scalar. The node
init uses `m_i² = ⟨p_i,p_i⟩` and `⟨p_i,P_jet⟩`, both invariant. If we also feed the **energy** `E_i` — which is
*not* invariant (a boost mixes E and p_z) — we puncture the guarantee: the network's output now depends on the
frame. We build exactly that "broken" model, boost the test jets, and measure the invariance violation
`max|f(Λx)−f(x)|` against the intact model. We also check the accuracy: energy is discriminating, so breaking
the symmetry can *raise* the in-distribution AUC a little — while forfeiting the exactness that matters when the
test frame differs from training.

In [ ]:
class LorentzNetBroken(nn.Module):
    """LorentzNet-lite but the node init also ingests the RAW energy E_i (a non-invariant) -> breaks invariance."""
    def __init__(self, hidden=64, n_layers=3):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, data):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
@torch.no_grad()
def boost_violation(model, xi=0.4):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 2 — Augment harder

> *Widen augmentation to full random Λ with larger boosts, and scan the residual drift |f(Λx)−f(x)| of the
> augmented model vs the built-in one. How close can augmentation get, and at what cost in epochs?*

**The concept.** The standing objection to equivariant nets: "just augment." We train the flexible DeepSets
baseline with **full random Lorentz augmentation** at several epoch budgets and measure two things: (i) test
AUC, and (ii) the residual **invariance drift** `max|f(Λx)−f(x)|` averaged over random Λ. The built-in
LorentzNet has drift at the float32 floor for *free*; augmentation can shrink DeepSets' drift, but only slowly
(more epochs), never to zero, and at a persistent AUC deficit. This quantifies "approximate, and you pay for
it".

In [ ]:
@torch.no_grad()
def mean_drift(model, nL=8, xi_scale=1.0):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 3 — More particles, frame-dependent graph

> *Raise NPART and switch the fully-connected graph to k-NN of Minkowski distance. Re-run the §5c scan: does a
> frame-dependent graph measurably break invariance?*

**The concept — with a subtlety worth its own paragraph.** LorentzNet uses a **fully-connected** graph
precisely because "all pairs" is the same in every frame. Replace it with a k-NN graph and you must ask: *is
the metric you rank neighbours by invariant?* Two cases:

- **k-NN by Minkowski distance** `⟨p_i−p_j, p_i−p_j⟩`. This scalar is **Lorentz-invariant**, so the *ranking*
  of neighbours — and hence the graph — is the **same in every frame**. Invariance is preserved (up to rare
  top-k tie-breaking on near-degenerate distances).
- **k-NN by a lab/Euclidean metric** (e.g. `Σ_μ (Δp_μ)²` or ΔR in the lab). This is **frame-dependent**, so a
  boost reshuffles the neighbours, the graph changes, and invariance **breaks**.

We build both k-NN variants (untrained, as in §5c — this is about the function class) and scan the output drift
vs boost strength against the fully-connected model.

In [ ]:
def knn_edges(x, batch, k, metric):
    """k-NN edge_index within each graph. metric: 'mink' (invariant) or 'eucl' (frame-dependent)."""
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
class LorentzNetGraph(LorentzNetLite):
    """LorentzNet whose graph is k-NN by a chosen metric instead of fully-connected."""
    def __init__(self, k=8, metric="mink", **kw):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, data):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
@torch.no_grad()
def drift_scan(model, xis):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 4 — PELICAN-style: use *all* the pairwise invariants

> *Replace the message inputs with the full pairwise invariant matrix ⟨p_i,p_j⟩ and a permutation-equivariant
> aggregator. Compare accuracy and parameter count to LorentzNet-lite.*

**The concept.** LorentzNet builds messages from *two* invariants per edge. **PELICAN** (Bogatskiy et al.)
takes the maximalist view: the input is the **complete** matrix of pairwise Lorentz invariants
`B_ij = ψ(⟨p_i,p_j⟩)` (its diagonal is each particle's mass²), and the network is a stack of
**permutation-equivariant** maps on that 2-D array, ending in an invariant readout. Since every entry is a
Lorentz scalar, the whole model is Lorentz-invariant *and* permutation-equivariant. We build a compact
PELICAN-lite: embed each matrix entry, aggregate each row with a permutation-invariant mean to node features,
then pool to a jet vector. (PELICAN proper uses the full set of permutation-equivariant aggregators; we use one
for a stable, minimal demo.)

In [ ]:
class PelicanLite(nn.Module):
    """Messages from the FULL pairwise invariant matrix ψ(<p_i,p_j>); permutation-equivariant row aggregation."""
    def __init__(self, hidden=32, n_classes=2):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, data):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 5 — Lorentz-equivariant regression

> *Switch on `update_coords` in `LGEB`, read out the evolved 4-vector, and regress the jet 4-momentum. Verify
> the prediction boosts correctly.*

**The concept.** Classification needs only *invariance*, so LorentzNet discarded LGEB's coordinate branch. But
that branch — `x_i' = x_i + c Σ_j φ_x(m_ij)(x_i−x_j)` — is an **equivariant vector field** on 4-momenta: it is
the only way to output something that *transforms* (a direction, a 4-momentum). We turn it on, pool the evolved
4-vectors into a predicted **jet 4-momentum**, and regress the true one. The headline check is not the loss but
the **equivariance**: `pred(Λx) = Λ pred(x)` to numerical precision — something an invariant-only network
*structurally cannot* satisfy (it can only emit numbers that don't transform).

In [ ]:
class Lorentz4VecRegressor(nn.Module):
    """LGEB with update_coords=ON; predicts a jet 4-momentum as an equivariant sum of evolved 4-vectors."""
    def __init__(self, hidden=64, n_layers=3):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, data):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
class InvariantRegressor(nn.Module):
    """Tries to emit a 4-vector from pooled invariants -> cannot be equivariant (parallel to §6b)."""
    def __init__(self, hidden=64):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, data):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
def train_reg(model, loader, epochs=EPOCHS, lr=1e-3):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
@torch.no_grad()
def reg_report(model):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 6 — Will a big model catch up?

> *Swap the DeepSets baseline for the Module-3 Transformer (no Lorentz bias) on the same 4-vectors and re-make
> the §6 curve. Does high capacity close the gap as data grows — and how much data?*

**The concept.** §6 used a deliberately simple DeepSets baseline that plateaus early — a critic could say "of
course equivariance wins, your baseline is weak." The fair test is a **high-capacity** non-equivariant model:
a Transformer (Module 3), all-to-all attention, no Lorentz bias, on the same 4-vectors. We remake the data-
efficiency curve: LorentzNet vs Transformer vs plain DeepSets at increasing training size. The question is
whether raw capacity + data eventually erases the inductive-bias advantage.

In [ ]:
from torch_geometric.utils import to_dense_batch
class TransformerBaseline(nn.Module):
    """High-capacity, permutation-invariant, NOT Lorentz-aware: MHA over 4-vectors + masked mean pool."""
    def __init__(self, dim=64, depth=3, heads=4, n_classes=2):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, data):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Exercise 7 — Grow the L-GATr

> *Add the geometric-product bilinear to §8e's MLP (multiply two `equi_linear` projections with `geo_prod`)
> and/or a scalar channel, and scan channels/blocks. Does the extra representation power lift the AUC toward
> LorentzNet's — and does the boost-invariance check still pass?*

**The concept.** The minimal L-GATr core used equivariant-linear maps and invariant-score attention, but its
"MLP" was only a gated nonlinearity — no way for different **grades** of a multivector to interact. The
**geometric product** `geo_prod` is exactly that missing bilinear: multiplying two equivariant-linear
projections mixes scalar↔vector↔bivector↔… while staying equivariant (the group action intertwines the
geometric product). We first rebuild the GA machinery, then add a **geometric bilinear** to each block and
compare AUC and the boost-invariance check to the minimal model.

In [ ]:
# --- geometric algebra Cl(1,3) core (from Module 4 §8a–8d), condensed --------------------
import itertools
GMET = [1.0, -1.0, -1.0, -1.0]
BLADES = [c for g in range(5) for c in itertools.combinations(range(4), g)]
BLADE_MASK = [sum(1 << i for i in b) for b in BLADES]; BIDX = {m: k for k, m in enumerate(BLADE_MASK)}
GRADE = torch.tensor([len(b) for b in BLADES])
def _pc(x): return bin(x).count("1")
def _bmul(A, B):
    swaps = sum(_pc(A & ~((1 << (j+1)) - 1) & 0xF) for j in range(4) if (B >> j) & 1)
    sign = -1.0 if swaps & 1 else 1.0; metric = 1.0
    for j in range(4):
        if ((A & B) >> j) & 1: metric *= GMET[j]
    return sign*metric, A ^ B
GP = torch.zeros(16, 16, 16)
for i, Ai in enumerate(BLADE_MASK):
    for j, Aj in enumerate(BLADE_MASK):
        co, res = _bmul(Ai, Aj); GP[i, j, BIDX[res]] += co
REV = torch.tensor([(-1.0)**(gg*(gg-1)//2) for gg in GRADE.tolist()])
IP = REV * GP[torch.arange(16), torch.arange(16), 0]
def geo_prod(x, y): return torch.einsum("...i,...j,ijk->...k", x, y, GP)
def embed_vec(v):
    mv = torch.zeros(*v.shape[:-1], 16, device=v.device); mv[..., 1:5] = v; return mv
def rho(Lam):
    M = torch.zeros(16, 16)
    for a, T in enumerate(BLADES):
        for b, S in enumerate(BLADES):
            if len(T) == len(S): M[a, b] = 1.0 if len(T) == 0 else torch.det(Lam[list(T)][:, list(S)])
    return M
def _commutant(transforms, tol=1e-4):
    I16 = torch.eye(16); rows = []
    for L in transforms:
        M = rho(L); rows.append(torch.kron(I16, M) - torch.kron(M.T.contiguous(), I16))
    _, S, Vh = torch.linalg.svd(torch.cat(rows, 0)); ndim = int((S < tol*S.max()).sum())
    return ndim, Vh[-ndim:].reshape(ndim, 16, 16)
dim_so, GA_LIN_BASIS = _commutant([random_lorentz() for _ in range(30)])
print("equivariant linear basis dim (SO+(1,3)):", dim_so, "(expected 10)")
def equi_linear(x, W):
    Bx = torch.einsum("kij,...cj->...cki", GA_LIN_BASIS, x)
    return torch.einsum("oik,...iku->...ou", W, Bx)
def gated_gelu(x): return F.gelu(x[..., 0:1]) * x
def ga_absnorm(x):
    per = IP * x * x
    return torch.stack([per[..., GRADE == g].sum(-1) for g in range(5)], -1).abs().sum(-1, keepdim=True)
def ga_layernorm(x, eps=0.01): return x * torch.rsqrt(ga_absnorm(x).mean(-2, keepdim=True).clamp_min(eps))
# move GA tensors to device
GP, IP, REV, GRADE, GA_LIN_BASIS = (t.to(device) for t in (GP, IP, REV, GRADE, GA_LIN_BASIS))

In [ ]:
class MiniLGATr(nn.Module):
    """Minimal (baseline) or +geometric-bilinear L-GATr classifier on jet 4-momenta."""
    def __init__(self, C=8, blocks=2, bilinear=False):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
    def forward(self, p4, mask):
        # TODO: implement this (see the exercise prompt above)
        raise NotImplementedError
def train_lgatr(model, epochs=12):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
@torch.no_grad()
def lgatr_auc(model):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError
@torch.no_grad()
def lgatr_boost_resid(model, xi=0.5):
    # TODO: implement this (see the exercise prompt above)
    raise NotImplementedError

# TODO: now instantiate the above, then train / evaluate / plot
#       using the helpers from Setup (e.g. train(...), evaluate(...), background_rejection(...)).

## Wrap-up

You've reached the end of the worksheet. If every exercise runs without a `NotImplementedError`, you've implemented them all — compare your results and reasoning with the solutions notebook. The through-line across the course: **every model is constrained message passing** — a choice of relations, a permutation-invariant aggregation, and a baked-in symmetry.